# BagZITboost — 전처리 PARAMS HPO (temp)

**목적**: 모델 HP 를 zit-final-100 best 로 고정하고 **전처리 PARAMS 만 Optuna 로 탐색**.
두 PC 병렬 실행 전략의 PP 파트.

**탐색 전략**:
- 모델 HP: zit-final-100 best (180 trial 결과) **FIXED_HP** 로 고정
- PP 탐색공간: missing/corr/spatial/indicator 임계값 7개 + τ_π = **8 HP**
- `corr_keep_by`, `post_impute_corr_keep_by`: **'std' 고정** (target_corr 는 fold leakage)
- N_TRIALS=30

**격리**: 출력 `4_output/_temp/bag_zit_pp_hpo/` (모델 노트북 `bag_zit_hpo` 와 별개).

## 1. 환경 + import

In [1]:
import os, sys

%run ../../../setup.py

import io, contextlib
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess
from modules.zi_tweedie import ZITboostRegressor

import lightgbm as lgb
from sklearn.model_selection import KFold
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
optuna v4.7.0


## 2. 실험 설정 (FIXED_HP = zit-final-100 best)

In [2]:
# ── 실험 식별 ──
EXP_ID  = 'bag-zit-pp-hpo-001'
USER    = 'jh'
N_TRIALS = 30
N_FOLDS  = 5
CLIP_Y_EXTREME = True

# ── τ_π 학습 (8번째 HP) ──
USE_DIE_PI_THRESHOLD = True

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_pp_hpo')
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')

# ── FIXED_HP: zit-final-100 best (4_output/final/zit_only/best_params.json) ──
# 180 trial Optuna 결과의 best HP. 전처리 탐색 동안 모델 HP 는 이 값에 고정.
FIXED_HP = dict(
    zeta                  = 1.149489889666854,
    n_em_iters            = 15,
    em_tol                = 1e-7,
    mu_n_estimators       = 164,
    mu_learning_rate      = 0.00738594709027388,
    mu_num_leaves         = 153,
    mu_max_depth          = 5,
    mu_min_child_samples  = 72,
    mu_subsample          = 0.6938934613616774,
    mu_colsample_bytree   = 0.35457351273995524,
    mu_reg_alpha          = 0.0001691645198209928,
    mu_reg_lambda         = 0.07334833575531088,
    pi_n_estimators       = 215,
    pi_learning_rate      = 0.07300634153014683,
    pi_num_leaves         = 69,
    pi_max_depth          = 8,
    pi_min_child_samples  = 44,
    phi_n_estimators      = 67,
    phi_learning_rate     = 0.012225381475545456,
    phi_num_leaves        = 72,
    phi_max_depth         = 5,
    phi_min_child_samples = 29,
    random_state          = SEED,
    n_jobs                = -1,
    verbose               = -1,
    device                = 'cpu',
)

print(f'EXP_ID={EXP_ID} | USER={USER} | N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS}')
print(f'USE_DIE_PI_THRESHOLD={USE_DIE_PI_THRESHOLD}')
print(f'OUT_DIR={OUT_DIR}')
print(f'FIXED_HP keys: {len(FIXED_HP)} (zit-final-100 best)')

EXP_ID=bag-zit-pp-hpo-001 | USER=jh | N_TRIALS=1 | N_FOLDS=5
USE_DIE_PI_THRESHOLD=True
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_pp_hpo
FIXED_HP keys: 26 (zit-final-100 best)


## 3. 데이터 로드 (전처리는 매 trial 다시 적용)

preprocess.run() 결과가 trial마다 다르므로 cell 안에서는 **load_all + Y 정리만** 하고,
numpy 변환과 전처리는 objective 안에서 수행.

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

print(f'\n[데이터 로드 완료]')
print(f'  xs: {xs.shape}, feat_cols: {len(feat_cols)}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개 샘플

[데이터 로드 완료]
  xs: (174572, 1091), feat_cols: 1087
  unit train=26,187, val=8,727, test=8,729


## 4. BagZITboostRegressor 정의 (모델 노트북과 동일)

In [4]:
class BagZITboostRegressor(ZITboostRegressor):
    """ZITboost + bag (unit) constraint via B3 allocation."""

    @staticmethod
    def _allocate_b3(unit_y_per_unit, contribution, inverse, n_units):
        contrib_sum_per_unit = np.zeros(n_units)
        np.add.at(contrib_sum_per_unit, inverse, contribution)
        contrib_sum_die = contrib_sum_per_unit[inverse]
        n_die_per_unit = np.bincount(inverse, minlength=n_units).astype(np.float64)
        n_die_die = n_die_per_unit[inverse]
        share = np.where(
            contrib_sum_die > 1e-12,
            contribution / np.maximum(contrib_sum_die, 1e-12),
            1.0 / np.maximum(n_die_die, 1.0),
        )
        return unit_y_per_unit[inverse] * share

    def fit(self, X, y, unit_id):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()
        unit_id = np.asarray(unit_id)

        unique_units, first_idx, inverse = np.unique(
            unit_id, return_index=True, return_inverse=True
        )
        n_units = len(unique_units)
        unit_y_per_unit = y[first_idx]

        n_die_per_unit = np.bincount(inverse, minlength=n_units).astype(np.float64)
        y_die_alloc = unit_y_per_unit[inverse] / np.maximum(n_die_per_unit[inverse], 1.0)

        pi_arr, mu_arr, phi_arr = self._initialize(X, y_die_alloc)
        self._phi_current = phi_arr

        self.em_history_ = []
        prev_rmse = np.inf
        em_iter = 0

        for em_iter in range(self.n_em_iters):
            if em_iter > 0:
                contribution = np.clip((1 - pi_arr) * mu_arr, 0, None)
                y_die_alloc = self._allocate_b3(
                    unit_y_per_unit, contribution, inverse, n_units
                )

            posterior = self._e_step(y_die_alloc, pi_arr, mu_arr, phi_arr)

            self._phi_current = phi_arr
            lgb_pi, lgb_mu, lgb_phi, pi_arr, mu_arr, phi_arr = \
                self._m_step(X, y_die_alloc, posterior)

            pred_die = np.clip((1 - pi_arr) * mu_arr, 0, None)
            pred_unit = np.zeros(n_units)
            np.add.at(pred_unit, inverse, pred_die)
            rmse_unit = float(np.sqrt(np.mean((unit_y_per_unit - pred_unit) ** 2)))

            self.em_history_.append({
                'iter':      em_iter + 1,
                'unit_rmse': rmse_unit,
                'pi_mean':   float(pi_arr.mean()),
                'mu_mean':   float(mu_arr.mean()),
            })

            rmse_delta = prev_rmse - rmse_unit
            if em_iter >= 2 and abs(rmse_delta) < self.em_tol:
                break
            prev_rmse = rmse_unit

        self.n_em_iters_actual_ = em_iter + 1
        self.lgb_pi_ = lgb_pi
        self.lgb_mu_ = lgb_mu
        self.lgb_phi_ = lgb_phi
        self.fitted_ = True
        return self

    def predict_unit(self, X, unit_id):
        if not hasattr(self, 'fitted_'):
            raise ValueError('Model not fitted')
        unit_id = np.asarray(unit_id)
        unique_units, inverse = np.unique(unit_id, return_inverse=True)
        n_units = len(unique_units)
        pred_die = self.predict(X)
        pred_unit = np.zeros(n_units)
        np.add.at(pred_unit, inverse, pred_die)
        return pred_unit, unique_units


print('BagZITboostRegressor 정의 완료')

BagZITboostRegressor 정의 완료


## 5. PP 탐색공간 정의 (7 PP HP + τ_π = 8 HP)

**고정**: `corr_keep_by='std'`, `post_impute_corr_keep_by='std'`, `corr_winsorize_pct=0.0`, `const_threshold=1e-6`

**탐색**:

| HP | 범위 | 기본값 | 설명 |
|---|---|---|---|
| missing_threshold | [0.20, 0.60] | 0.4 | 결측률 컷오프 |
| corr_threshold | [0.85, 0.99] | 0.90 | 1차 \|r\| 컷오프 |
| add_indicator | True/False | True | 결측 indicator 컬럼 |
| indicator_threshold | [0.01, 0.20] | 0.05 | indicator 생성 기준 |
| spatial_max_dist | {1, 2, 3, 4, 5} | 5.0 | spatial 보간 거리 (정수 단위, search_space.py 표준) |
| post_impute_corr_threshold | [0.95, 0.999] | 0.98 | 2차 \|r\| 컷오프 |
| **τ_π** | **[0.5, 1.0]** | — | π > τ_π 인 die pred=0 |

In [5]:
def suggest_pp_params(trial):
    """전처리 PARAMS 탐색공간 + τ_π.

    Returns
    -------
    (pp_params, tau_pi) : (dict, float)
    """
    pp_params = dict(
        missing_threshold          = trial.suggest_float('missing_threshold', 0.20, 0.60),
        corr_threshold             = trial.suggest_float('corr_threshold', 0.85, 0.99),
        corr_keep_by               = 'std',                                                    # ★ 고정 (fold leakage 방지)
        add_indicator              = trial.suggest_categorical('add_indicator', [True, False]),
        indicator_threshold        = trial.suggest_float('indicator_threshold', 0.01, 0.20),
        spatial_max_dist           = trial.suggest_categorical('spatial_max_dist', [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0]),
        post_impute_corr_threshold = trial.suggest_float('post_impute_corr_threshold', 0.95, 0.999),
        post_impute_corr_keep_by   = 'std',                                                    # ★ 고정
    )

    if USE_DIE_PI_THRESHOLD:
        tau_pi = trial.suggest_float('tau_pi', 0.5, 1.0)
    else:
        tau_pi = 1.0

    return pp_params, tau_pi


print(f'PP 탐색공간 {7 if USE_DIE_PI_THRESHOLD else 6} HP + τ_π 정의 완료')

PP 탐색공간 7 HP + τ_π 정의 완료


## 6. K-fold objective (preprocess inside trial)

각 trial:
1. PP params + τ_π suggest
2. **preprocess.run() 호출 (출력 silenced)** → 매 trial 다른 결과
3. numpy 변환
4. 5-fold 학습 (FIXED_HP) → die-level π/μ → τ_π 적용 → unit sum → RMSE

In [6]:
import time

unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))


def _sum_die_to_unit(pred_die, uid_die):
    """die-level pred 를 unit별 합산."""
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    n_units = len(unique_units)
    pred_unit = np.zeros(n_units)
    np.add.at(pred_unit, inverse, pred_die)
    return pred_unit, unique_units


def _apply_tau_pi(pred_die, pi_die, tau_pi):
    return np.where(pi_die > tau_pi, 0.0, pred_die)


def _run_preprocess_silenced(pp_params):
    """preprocess.run() 출력 silenced. (pp_dict, n_feats) 반환."""
    with contextlib.redirect_stdout(io.StringIO()):
        pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=pp_params)
    return pp


def objective(trial):
    pp_params, tau_pi = suggest_pp_params(trial)

    t0 = time.time()

    # ── 전처리 (silenced) ──
    try:
        pp = _run_preprocess_silenced(pp_params)
    except Exception as e:
        print(f'  trial #{trial.number}: preprocess 실패 — {e}')
        raise optuna.TrialPruned()

    feat_cols_clean = pp['feat_cols']
    if len(feat_cols_clean) < 50:
        print(f'  trial #{trial.number}: feat_cols={len(feat_cols_clean)} 너무 적음 — pruned')
        trial.set_user_attr('pruned_reason', 'too_few_features')
        trial.set_user_attr('n_feats', len(feat_cols_clean))
        raise optuna.TrialPruned()

    xs_train_die = pp['xs_train']
    xs_val_die   = pp['xs_val']
    xs_test_die  = pp['xs_test']

    # numpy 변환
    X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
    X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
    X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
    uid_train_die = xs_train_die[KEY_COL].values
    uid_val_die   = xs_val_die[KEY_COL].values
    uid_test_die  = xs_test_die[KEY_COL].values
    y_train_die_broadcast = xs_train_die[KEY_COL].map(y_train_unit).values.astype(np.float64)

    pp_time = time.time() - t0

    # ── 5-fold 학습 (FIXED_HP) ──
    oof_unit_pred  = pd.Series(np.nan, index=y_train_unit.index)
    fold_preds_val = []
    fold_preds_test = []
    fold_oof_rmse  = []

    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unit_ids_train_unique[tr_uidx]
        vl_units = unit_ids_train_unique[vl_uidx]
        tr_die_mask = np.isin(uid_train_die, tr_units)
        vl_die_mask = np.isin(uid_train_die, vl_units)

        X_tr   = X_train_die[tr_die_mask]
        X_vl   = X_train_die[vl_die_mask]
        y_tr   = y_train_die_broadcast[tr_die_mask]
        uid_tr = uid_train_die[tr_die_mask]
        uid_vl = uid_train_die[vl_die_mask]

        model = BagZITboostRegressor(**FIXED_HP)
        model.fit(X_tr, y_tr, unit_id=uid_tr)

        # OOF — die-level π/μ → τ_π 적용 → unit sum
        pi_vl, mu_vl, _ = model.predict_components(X_vl)
        pred_die_vl_raw = np.clip((1 - pi_vl) * mu_vl, 0, None)
        pred_die_vl = _apply_tau_pi(pred_die_vl_raw, pi_vl, tau_pi)
        pred_unit_vl, ids_vl = _sum_die_to_unit(pred_die_vl, uid_vl)

        oof_unit_pred.loc[ids_vl] = pred_unit_vl
        y_vl_true = y_train_unit.loc[ids_vl].values
        fold_rmse = float(np.sqrt(np.mean((pred_unit_vl - y_vl_true) ** 2)))
        fold_oof_rmse.append(fold_rmse)

        # holdout val/test
        pi_v, mu_v, _ = model.predict_components(X_val_die)
        pred_die_v_raw = np.clip((1 - pi_v) * mu_v, 0, None)
        pred_die_v = _apply_tau_pi(pred_die_v_raw, pi_v, tau_pi)
        pred_unit_val, ids_val = _sum_die_to_unit(pred_die_v, uid_val_die)

        pi_t, mu_t, _ = model.predict_components(X_test_die)
        pred_die_t_raw = np.clip((1 - pi_t) * mu_t, 0, None)
        pred_die_t = _apply_tau_pi(pred_die_t_raw, pi_t, tau_pi)
        pred_unit_test, ids_test = _sum_die_to_unit(pred_die_t, uid_test_die)

        fold_preds_val.append(pd.Series(pred_unit_val, index=ids_val))
        fold_preds_test.append(pd.Series(pred_unit_test, index=ids_test))

        # Pruning report
        avg_so_far = float(np.mean(fold_oof_rmse))
        trial.report(avg_so_far, step=fold_idx)
        if trial.should_prune():
            elapsed = time.time() - t0
            trial.set_user_attr('pruned_at_fold', fold_idx + 1)
            trial.set_user_attr('elapsed_sec', elapsed)
            trial.set_user_attr('tau_pi', tau_pi)
            trial.set_user_attr('n_feats', len(feat_cols_clean))
            raise optuna.TrialPruned()

    val_unit_pred  = pd.concat(fold_preds_val,  axis=1).mean(axis=1).reindex(y_val_unit.index)
    test_unit_pred = pd.concat(fold_preds_test, axis=1).mean(axis=1).reindex(y_test_unit.index)

    if oof_unit_pred.isna().any():
        raise RuntimeError('OOF NaN')

    oof_rmse  = float(np.sqrt(np.mean((oof_unit_pred.values  - y_train_unit.values) ** 2)))
    val_rmse  = float(np.sqrt(np.mean((val_unit_pred.values  - y_val_unit.values)  ** 2)))
    test_rmse = float(np.sqrt(np.mean((test_unit_pred.values - y_test_unit.values) ** 2)))

    elapsed = time.time() - t0
    trial.set_user_attr('val_rmse',     val_rmse)
    trial.set_user_attr('test_rmse',    test_rmse)
    trial.set_user_attr('fold_oof_rmse', fold_oof_rmse)
    trial.set_user_attr('elapsed_sec',  elapsed)
    trial.set_user_attr('pp_time_sec',  pp_time)
    trial.set_user_attr('tau_pi',       tau_pi)
    trial.set_user_attr('n_feats',      len(feat_cols_clean))

    print(f'  trial #{trial.number}: τ_π={tau_pi:.3f}, n_feats={len(feat_cols_clean)}, '
          f'oof={oof_rmse:.6f}, val={val_rmse:.6f}, test={test_rmse:.6f}, '
          f'pp={pp_time:.0f}s+fit={elapsed-pp_time:.0f}s')

    return oof_rmse


print(f'fold split 고정: {N_FOLDS} folds, {len(FOLDS)} 개')

fold split 고정: 5 folds, 5 개


## 7. Optuna study 실행

샘플러: TPE multivariate (PP HP joint 분포 학습)
Pruner: MedianPruner (n_startup=4, n_warmup=2)
DB: SQLite (재개 가능)

In [7]:
sampler = TPESampler(
    multivariate=True,
    group=True,
    n_startup_trials=4,
    seed=SEED,
)
pruner = MedianPruner(
    n_startup_trials=4,
    n_warmup_steps=2,
)

study_meta = {
    'exp_id':              EXP_ID,
    'user':                USER,
    'model':               'BagZITboost (FIXED_HP=zit-final-100 best) + PP HPO',
    'n_trials':            N_TRIALS,
    'n_folds':             N_FOLDS,
    'use_die_pi_threshold': USE_DIE_PI_THRESHOLD,
    'fixed_hp_source':     '4_output/final/zit_only/best_params.json',
    'sampler':             'TPE multivariate group=True n_startup=4',
    'pruner':              'MedianPruner n_startup=4 n_warmup=2',
    'CLIP_Y_EXTREME':      CLIP_Y_EXTREME,
    'SEED':                int(SEED),
    'fixed_hp':            {k: v for k, v in FIXED_HP.items() if k not in ['random_state', 'n_jobs', 'verbose', 'device']},
}

study = optuna.create_study(
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    sampler=sampler,
    pruner=pruner,
    direction='minimize',
    load_if_exists=True,
)
for k, v in study_meta.items():
    study.set_user_attr(k, str(v))

print(f'study: {study.study_name}, DB: {DB_PATH}')
print(f'기존 trial 수: {len(study.trials)}')

t_total = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'\n[HPO 완료] 전체 {time.time()-t_total:.0f}s, total trials={len(study.trials)}')
print(f'  best OOF RMSE: {study.best_value:.6f}')

[I 2026-04-30 14:34:53,452] A new study created in RDB with name: bag-zit-pp-hpo-001


study: bag-zit-pp-hpo-001, DB: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_pp_hpo\optuna_jh_bag-zit-pp-hpo-001.db
기존 trial 수: 0


  0%|          | 0/1 [00:00<?, ?it/s]

  trial #0: τ_π=0.985, n_feats=702, oof=0.005501, val=0.005709, test=0.008412, pp=39s+fit=1138s
[I 2026-04-30 14:54:31,225] Trial 0 finished with value: 0.005501187204426149 and parameters: {'missing_threshold': 0.349816047538945, 'corr_threshold': 0.9831000028973882, 'add_indicator': True, 'indicator_threshold': 0.039643541684062936, 'spatial_max_dist': 6.0, 'post_impute_corr_threshold': 0.9510086402204943, 'tau_pi': 0.9849549260809971}. Best is trial 0 with value: 0.005501187204426149.

[HPO 완료] 전체 1178s, total trials=1
  best OOF RMSE: 0.005501


## 8. Best params + trial history

In [8]:
import matplotlib.pyplot as plt

best_trial = study.best_trial
best_params = best_trial.params

print(f'=== Best Trial #{best_trial.number} ===')
print(f'  OOF RMSE  : {best_trial.value:.6f}')
print(f'  val RMSE  : {best_trial.user_attrs.get("val_rmse", "N/A")}')
print(f'  test RMSE : {best_trial.user_attrs.get("test_rmse", "N/A")}')
print(f'  best τ_π  : {best_trial.user_attrs.get("tau_pi", "N/A")}')
print(f'  n_feats   : {best_trial.user_attrs.get("n_feats", "N/A")}')
print(f'  elapsed   : {best_trial.user_attrs.get("elapsed_sec", 0):.0f}s')
print(f'\n  best PP params:')
for k, v in sorted(best_params.items()):
    print(f'    {k}: {v}')

df_trials = pd.DataFrame([
    {
        'trial':     t.number,
        'state':     t.state.name,
        'oof_rmse':  t.value,
        'val_rmse':  t.user_attrs.get('val_rmse'),
        'test_rmse': t.user_attrs.get('test_rmse'),
        'tau_pi':    t.user_attrs.get('tau_pi'),
        'n_feats':   t.user_attrs.get('n_feats'),
        'elapsed':   t.user_attrs.get('elapsed_sec'),
    }
    for t in study.trials
])
print('\n=== trial history ===')
print(df_trials.to_string(index=False))

complete = df_trials[df_trials['state'] == 'COMPLETE']
if len(complete) > 1:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(complete['trial'], complete['oof_rmse'], marker='o', label='OOF')
    if complete['val_rmse'].notna().any():
        ax.plot(complete['trial'], complete['val_rmse'], marker='s', alpha=0.7, label='val')
    ax.axhline(study.best_value, color='red', linestyle='--', alpha=0.5, label=f'best={study.best_value:.6f}')
    ax.set_xlabel('trial')
    ax.set_ylabel('unit RMSE')
    ax.set_title('PP HPO trial history')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

=== Best Trial #0 ===
  OOF RMSE  : 0.005501
  val RMSE  : 0.005709114414433799
  test RMSE : 0.00841226335692651
  best τ_π  : 0.9849549260809971
  n_feats   : 702
  elapsed   : 1177s

  best PP params:
    add_indicator: True
    corr_threshold: 0.9831000028973882
    indicator_threshold: 0.039643541684062936
    missing_threshold: 0.349816047538945
    post_impute_corr_threshold: 0.9510086402204943
    spatial_max_dist: 6.0
    tau_pi: 0.9849549260809971

=== trial history ===
 trial    state  oof_rmse  val_rmse  test_rmse   tau_pi  n_feats     elapsed
     0 COMPLETE  0.005501  0.005709   0.008412 0.984955      702 1177.376696


## 8.5 PP HP importance + 그룹별 분석

fANOVA 기반 단일 HP importance + 분류:
- **missing**: missing_threshold, add_indicator, indicator_threshold
- **corr**: corr_threshold, post_impute_corr_threshold
- **spatial**: spatial_max_dist
- **τ_π**: tau_pi

In [9]:
import optuna.importance as opt_imp

try:
    imp = opt_imp.get_param_importances(study)

    print('=== HP importance (fANOVA) ===')
    for i, (k, v) in enumerate(sorted(imp.items(), key=lambda x: -x[1])):
        bar = '█' * int(v * 50)
        print(f'  {i+1:2d}. {k:34s}: {v:.4f}  {bar}')

    # 그룹별 합산
    groups = {'missing': 0.0, 'corr': 0.0, 'spatial': 0.0, 'τ_π': 0.0}
    for k, v in imp.items():
        if k.startswith('missing') or k in ('add_indicator', 'indicator_threshold'):
            groups['missing'] += v
        elif 'corr_threshold' in k:
            groups['corr'] += v
        elif k.startswith('spatial'):
            groups['spatial'] += v
        elif k == 'tau_pi':
            groups['τ_π'] += v

    print('\n=== 그룹별 importance 합산 ===')
    for g, v in sorted(groups.items(), key=lambda x: -x[1]):
        bar = '█' * int(v * 50)
        print(f'  {g:10s}: {v:.4f}  {bar}')

    try:
        import optuna.visualization as ov
        ov.plot_param_importances(study).show()
        ov.plot_parallel_coordinate(study).show()
        top2 = list(sorted(imp.items(), key=lambda x: -x[1]))[:2]
        if len(top2) >= 2:
            ov.plot_contour(study, params=[top2[0][0], top2[1][0]]).show()
    except Exception as e:
        print(f'\nplotly 시각화 스킵: {e}')

except (RuntimeError, ValueError) as e:
    print(f'importance 분석 스킵: {e}')
    print('  → trial 수가 적거나 모든 trial 이 같은 값일 가능성.')

importance 분석 스킵: Cannot evaluate parameter importances with only a single trial.
  → trial 수가 적거나 모든 trial 이 같은 값일 가능성.


## 9. Best PP 5-fold refit + die-level 캡처

best PP params 로 다시 preprocess → FIXED_HP 로 5-fold 학습 → die-level π/μ/pred 캡처.

In [10]:
best_oof_rmse  = float(best_trial.value)
best_val_rmse  = float(best_trial.user_attrs.get('val_rmse', np.nan))
best_test_rmse = float(best_trial.user_attrs.get('test_rmse', np.nan))
best_tau_pi    = float(best_trial.user_attrs.get('tau_pi', 1.0))

print('=== Best (5-fold OOF + holdout) ===')
print(f'  OOF unit RMSE  : {best_oof_rmse:.6f}')
print(f'  val unit RMSE  : {best_val_rmse:.6f}')
print(f'  test unit RMSE : {best_test_rmse:.6f}')
print(f'  best τ_π       : {best_tau_pi:.4f}')

# best PP params 만 추출 (tau_pi 제외)
best_pp_params = {k: v for k, v in best_params.items() if k != 'tau_pi'}
best_pp_params['corr_keep_by'] = 'std'              # 고정값
best_pp_params['post_impute_corr_keep_by'] = 'std'  # 고정값

print(f'\n=== best PP params 로 preprocess 재실행 ===')
for k, v in sorted(best_pp_params.items()):
    print(f'  {k}: {v}')

pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=best_pp_params)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values
y_train_die_broadcast = xs_train_die[KEY_COL].map(y_train_unit).values.astype(np.float64)

print(f'\n  best 전처리 후 feat_cols: {len(feat_cols_clean)}')

# die-level 캡처용
n_train_die = len(X_train_die)
n_val_die   = len(X_val_die)
n_test_die  = len(X_test_die)

oof_die_pi   = np.full(n_train_die, np.nan)
oof_die_mu   = np.full(n_train_die, np.nan)
oof_die_pred = np.full(n_train_die, np.nan)

val_die_pi   = np.zeros(n_val_die)
val_die_mu   = np.zeros(n_val_die)
val_die_pred = np.zeros(n_val_die)

test_die_pi   = np.zeros(n_test_die)
test_die_mu   = np.zeros(n_test_die)
test_die_pred = np.zeros(n_test_die)

print('\n=== FIXED_HP 로 5-fold refit (die-level 캡처) ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unit_ids_train_unique[tr_uidx]
    vl_units = unit_ids_train_unique[vl_uidx]
    tr_die_mask = np.isin(uid_train_die, tr_units)
    vl_die_mask = np.isin(uid_train_die, vl_units)
    X_tr, y_tr = X_train_die[tr_die_mask], y_train_die_broadcast[tr_die_mask]
    X_vl       = X_train_die[vl_die_mask]
    uid_tr     = uid_train_die[tr_die_mask]

    model = BagZITboostRegressor(**FIXED_HP)
    model.fit(X_tr, y_tr, unit_id=uid_tr)

    pi_vl, mu_vl, _ = model.predict_components(X_vl)
    pred_vl = np.clip((1 - pi_vl) * mu_vl, 0, None)
    oof_die_pi[vl_die_mask]   = pi_vl
    oof_die_mu[vl_die_mask]   = mu_vl
    oof_die_pred[vl_die_mask] = pred_vl

    pi_v, mu_v, _ = model.predict_components(X_val_die)
    pi_t, mu_t, _ = model.predict_components(X_test_die)
    val_die_pi   += pi_v / N_FOLDS
    val_die_mu   += mu_v / N_FOLDS
    val_die_pred += np.clip((1 - pi_v) * mu_v, 0, None) / N_FOLDS
    test_die_pi   += pi_t / N_FOLDS
    test_die_mu   += mu_t / N_FOLDS
    test_die_pred += np.clip((1 - pi_t) * mu_t, 0, None) / N_FOLDS

    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die_pi).any(),   'OOF die π 미커버'
assert not np.isnan(oof_die_mu).any(),   'OOF die μ 미커버'
assert not np.isnan(oof_die_pred).any(), 'OOF die pred 미커버'

print('\n[refit 완료] die-level π/μ/pred 캡처 OK')
print(f'  oof_die: {n_train_die}, val_die: {n_val_die}, test_die: {n_test_die}')

=== Best (5-fold OOF + holdout) ===
  OOF unit RMSE  : 0.005501
  val unit RMSE  : 0.005709
  test unit RMSE : 0.008412
  best τ_π       : 0.9850

=== best PP params 로 preprocess 재실행 ===
  add_indicator: True
  corr_keep_by: std
  corr_threshold: 0.9831000028973882
  indicator_threshold: 0.039643541684062936
  missing_threshold: 0.349816047538945
  post_impute_corr_keep_by: std
  post_impute_corr_threshold: 0.9510086402204943
  spatial_max_dist: 6.0
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=35%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9831000028973882, keep_by=std (std)
  제거: 116개, 잔여: 780개
    컬럼: 896 → 780 (116개 제거)
    DataFrame: (104748, 838)

[결측 indicator] 9개 컬럼 추가 (결

## 10. τ_π 적용 + raw vs clipped 비교

In [11]:
oof_die_pred_clipped  = np.where(oof_die_pi  > best_tau_pi, 0.0, oof_die_pred)
val_die_pred_clipped  = np.where(val_die_pi  > best_tau_pi, 0.0, val_die_pred)
test_die_pred_clipped = np.where(test_die_pi > best_tau_pi, 0.0, test_die_pred)

oof_unit_raw_arr,  oof_unit_ids  = _sum_die_to_unit(oof_die_pred,         uid_train_die)
oof_unit_clip_arr, _              = _sum_die_to_unit(oof_die_pred_clipped, uid_train_die)
val_unit_raw_arr,  val_unit_ids  = _sum_die_to_unit(val_die_pred,         uid_val_die)
val_unit_clip_arr, _              = _sum_die_to_unit(val_die_pred_clipped, uid_val_die)
test_unit_raw_arr, test_unit_ids = _sum_die_to_unit(test_die_pred,        uid_test_die)
test_unit_clip_arr, _             = _sum_die_to_unit(test_die_pred_clipped, uid_test_die)

oof_unit_raw   = pd.Series(oof_unit_raw_arr,  index=oof_unit_ids).reindex(y_train_unit.index)
oof_unit_clip  = pd.Series(oof_unit_clip_arr, index=oof_unit_ids).reindex(y_train_unit.index)
val_unit_raw   = pd.Series(val_unit_raw_arr,  index=val_unit_ids).reindex(y_val_unit.index)
val_unit_clip  = pd.Series(val_unit_clip_arr, index=val_unit_ids).reindex(y_val_unit.index)
test_unit_raw  = pd.Series(test_unit_raw_arr, index=test_unit_ids).reindex(y_test_unit.index)
test_unit_clip = pd.Series(test_unit_clip_arr, index=test_unit_ids).reindex(y_test_unit.index)

def _rmse(pred, true):
    return float(np.sqrt(np.mean((pred.values - true.values) ** 2)))

oof_rmse_raw      = _rmse(oof_unit_raw,   y_train_unit)
oof_rmse_clipped  = _rmse(oof_unit_clip,  y_train_unit)
val_rmse_raw      = _rmse(val_unit_raw,   y_val_unit)
val_rmse_clipped  = _rmse(val_unit_clip,  y_val_unit)
test_rmse_raw     = _rmse(test_unit_raw,  y_test_unit)
test_rmse_clipped = _rmse(test_unit_clip, y_test_unit)

print('=' * 75)
print(f'  best τ_π = {best_tau_pi:.4f}'
      + ('  (≈ off)' if best_tau_pi >= 0.99 else ''))
print('=' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"Raw":12s}  {oof_rmse_raw:11.6f}  {val_rmse_raw:11.6f}  {test_rmse_raw:11.6f}')
print(f'  {"Clipped":12s}  {oof_rmse_clipped:11.6f}  {val_rmse_clipped:11.6f}  {test_rmse_clipped:11.6f}')
print(f'  {"Δ":12s}  {oof_rmse_clipped-oof_rmse_raw:+11.6f}  '
      f'{val_rmse_clipped-val_rmse_raw:+11.6f}  '
      f'{test_rmse_clipped-test_rmse_raw:+11.6f}')
print('=' * 75)

if val_rmse_clipped > val_rmse_raw and test_rmse_clipped > test_rmse_raw:
    print('  ⚠ val/test 모두 악화 — τ_π 적용 비추천')
elif val_rmse_clipped < val_rmse_raw and test_rmse_clipped < test_rmse_raw:
    print('  ✅ val/test 모두 개선 — τ_π 채택 권장')
else:
    print('  ⚠ val/test 결과 엇갈림 — 신중 판단')

killed_oof  = float((oof_die_pi  > best_tau_pi).mean())
killed_val  = float((val_die_pi  > best_tau_pi).mean())
killed_test = float((test_die_pi > best_tau_pi).mean())
print(f'\n  τ_π 로 0 처리된 die 비율: OOF={killed_oof:.1%}, val={killed_val:.1%}, test={killed_test:.1%}')

  best τ_π = 0.9850
                        OOF          val         test
  Raw              0.005501     0.005709     0.008412
  Clipped          0.005501     0.005709     0.008412
  Δ               +0.000000    -0.000000    +0.000000
  ⚠ val/test 결과 엇갈림 — 신중 판단

  τ_π 로 0 처리된 die 비율: OOF=0.0%, val=0.0%, test=0.0%


## 11. 아티팩트 저장

In [12]:
import json

def _build_die_df(uid_arr, die_id_arr, position_arr, pi, mu, pred, pred_clipped, y_unit):
    df = pd.DataFrame({
        KEY_COL:        uid_arr,
        DIE_KEY_COL:    die_id_arr,
        'position':     position_arr,
        'pi':           pi,
        'one_minus_pi': 1.0 - pi,
        'mu':           mu,
        'pred':         pred,
        'pred_clipped': pred_clipped,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

oof_die_df = _build_die_df(
    uid_train_die, xs_train_die[DIE_KEY_COL].values, xs_train_die['position'].values,
    oof_die_pi, oof_die_mu, oof_die_pred, oof_die_pred_clipped, y_train_unit,
)
val_die_df = _build_die_df(
    uid_val_die, xs_val_die[DIE_KEY_COL].values, xs_val_die['position'].values,
    val_die_pi, val_die_mu, val_die_pred, val_die_pred_clipped, y_val_unit,
)
test_die_df = _build_die_df(
    uid_test_die, xs_test_die[DIE_KEY_COL].values, xs_test_die['position'].values,
    test_die_pi, test_die_mu, test_die_pred, test_die_pred_clipped, y_test_unit,
)
oof_die_df.to_csv(os.path.join(OUT_DIR,  'oof_die.csv'),  index=False)
val_die_df.to_csv(os.path.join(OUT_DIR,  'val_die.csv'),  index=False)
test_die_df.to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

def _build_unit_df(unit_pred, y_unit):
    return pd.DataFrame({
        KEY_COL:  unit_pred.index.values,
        'pred':   unit_pred.values,
        'health': y_unit.reindex(unit_pred.index).values,
    })

_build_unit_df(oof_unit_raw,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(val_unit_raw,  y_val_unit  ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(test_unit_raw, y_test_unit ).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

_build_unit_df(oof_unit_clip,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit_clipped.csv'),  index=False)
_build_unit_df(val_unit_clip,  y_val_unit  ).to_csv(os.path.join(OUT_DIR, 'val_unit_clipped.csv'),  index=False)
_build_unit_df(test_unit_clip, y_test_unit ).to_csv(os.path.join(OUT_DIR, 'test_unit_clipped.csv'), index=False)

with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'best_trial_number': best_trial.number,
        'best_oof_rmse':     best_oof_rmse,
        'best_val_rmse':     best_val_rmse,
        'best_test_rmse':    best_test_rmse,
        'best_tau_pi':       best_tau_pi,
        'best_pp_params':    best_pp_params,
        'fixed_hp':          {k: v for k, v in FIXED_HP.items() if k not in ['random_state', 'n_jobs', 'verbose', 'device']},
        'fixed_hp_source':   '4_output/final/zit_only/best_params.json',
    }, f, indent=2, ensure_ascii=False, default=str)

meta = {
    'exp_id':              EXP_ID,
    'model':               'BagZITboost (FIXED_HP) + PP HPO + τ_π',
    'use_die_pi_threshold': USE_DIE_PI_THRESHOLD,
    'best_tau_pi':         best_tau_pi,
    'n_trials_run':        len(study.trials),
    'n_trials_complete':   sum(1 for t in study.trials if t.state.name == 'COMPLETE'),
    'n_folds':             N_FOLDS,
    'best_pp_params':      best_pp_params,
    'best_n_feats':        len(feat_cols_clean),
    'raw_oof_rmse':        oof_rmse_raw,
    'raw_val_rmse':        val_rmse_raw,
    'raw_test_rmse':       test_rmse_raw,
    'clipped_oof_rmse':    oof_rmse_clipped,
    'clipped_val_rmse':    val_rmse_clipped,
    'clipped_test_rmse':   test_rmse_clipped,
    'study_best_oof_rmse': best_oof_rmse,
    'study_best_val_rmse': best_val_rmse,
    'study_best_test_rmse': best_test_rmse,
    'CLIP_Y_EXTREME':      CLIP_Y_EXTREME,
    'SEED':                int(SEED),
    'sampler':             'TPE multivariate group=True n_startup=4',
    'tau_pi_range':        '[0.5, 1.0]' if USE_DIE_PI_THRESHOLD else 'off',
    'fixed_hp_source':     '4_output/final/zit_only/best_params.json',
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:35s}  {sz:10,.1f} KB')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_pp_hpo
  best_params.json                            1.3 KB
  meta.json                                   1.2 KB
  oof_die.csv                            14,244.3 KB
  oof_unit.csv                              971.4 KB
  oof_unit_clipped.csv                      971.3 KB
  optuna_jh_bag-zit-pp-hpo-001.db           112.0 KB
  test_die.csv                            4,747.5 KB
  test_unit.csv                             323.8 KB
  test_unit_clipped.csv                     323.8 KB
  val_die.csv                             4,746.4 KB
  val_unit.csv                              323.6 KB
  val_unit_clipped.csv                      323.6 KB


## 12. 요약

In [13]:
print('=' * 75)
print(' BagZITboost PP HPO + τ_π — 결과 요약')
print('=' * 75)
print(f'  EXP_ID            : {EXP_ID}')
print(f'  N_TRIALS run      : {len(study.trials)}')
print(f'  완료 trial        : {sum(1 for t in study.trials if t.state.name == "COMPLETE")}')
print(f'  Best trial        : #{best_trial.number}')
print(f'  Best τ_π          : {best_tau_pi:.4f}'
      + ('  (≈ off)' if best_tau_pi >= 0.99 else ''))
print(f'  Best n_feats      : {len(feat_cols_clean)}')
print('-' * 75)
print(f'  {"":10s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"Raw":10s}  {oof_rmse_raw:11.6f}  {val_rmse_raw:11.6f}  {test_rmse_raw:11.6f}')
print(f'  {"Clipped":10s}  {oof_rmse_clipped:11.6f}  {val_rmse_clipped:11.6f}  {test_rmse_clipped:11.6f}')
print(f'  {"Δ":10s}  {oof_rmse_clipped-oof_rmse_raw:+11.6f}  '
      f'{val_rmse_clipped-val_rmse_raw:+11.6f}  '
      f'{test_rmse_clipped-test_rmse_raw:+11.6f}')
print('-' * 75)
print(f'  비교 기준선:')
print(f'    BagZIT baseline (HP+PP fixed):   OOF=0.005524, val=0.005729, test=0.008428')
print(f'    ZIT 단독 zit-final-100:          OOF=0.005501 (180 trials, 본 노트북 FIXED_HP 출처)')
print(f'    BagZIT HPO 002 (HP 흔듦):        병렬 실행 결과와 비교')
print('=' * 75)
print(f'  → PP만 흔들었을 때 개선폭 = (FIXED HP + best PP) vs (FIXED HP + 기본 PP)')

 BagZITboost PP HPO + τ_π — 결과 요약
  EXP_ID            : bag-zit-pp-hpo-001
  N_TRIALS run      : 1
  완료 trial        : 1
  Best trial        : #0
  Best τ_π          : 0.9850
  Best n_feats      : 702
---------------------------------------------------------------------------
                      OOF          val         test
  Raw            0.005501     0.005709     0.008412
  Clipped        0.005501     0.005709     0.008412
  Δ             +0.000000    -0.000000    +0.000000
---------------------------------------------------------------------------
  비교 기준선:
    BagZIT baseline (HP+PP fixed):   OOF=0.005524, val=0.005729, test=0.008428
    ZIT 단독 zit-final-100:          OOF=0.005501 (180 trials, 본 노트북 FIXED_HP 출처)
    BagZIT HPO 002 (HP 흔듦):        병렬 실행 결과와 비교
  → PP만 흔들었을 때 개선폭 = (FIXED HP + best PP) vs (FIXED HP + 기본 PP)
